In [1]:
import os
import sys
import pandas as pd


%pwd
os.chdir("../")

# Add the absolute path to src/ so Python can find automatch
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.append(src_path)
    
%pwd


'c:\\Users\\fiscarelli\\Desktop\\Progetti\\Manpower IT\\OnBench\\On-Bench-Candidates-to-Jobs-Auto-Match'

In [2]:
%pwd


'c:\\Users\\fiscarelli\\Desktop\\Progetti\\Manpower IT\\OnBench\\On-Bench-Candidates-to-Jobs-Auto-Match'

In [3]:
from src.onBench.utils.snowflake_utils import get_snowpark_session
session = get_snowpark_session()

[2026-06-26 17:07:31,185: INFO: connection: Snowflake Connector for Python Version: 3.7.0, Python Version: 3.10.11, Platform: Windows-10-10.0.26200-SP0]
[2026-06-26 17:07:31,186: INFO: connection: This connection is in OCSP Fail Open Mode. TLS Certificates would be checked for validity and revocation status. Any other Certificate Revocation related exceptions or OCSP Responder failures would be disregarded in favor of connectivity.]
[2026-06-26 17:07:34,819: INFO: session: Snowpark Session information: 
"version" : 1.15.0,
"python.version" : 3.10.11,
"python.connector.version" : 3.7.0,
"python.connector.session.id" : 126482891949514,
"os.name" : Windows
]
[2026-06-26 17:07:34,819: INFO: cursor: query: [use database IT_DISCOVERY]]
[2026-06-26 17:07:35,336: INFO: cursor: query execution done]
[2026-06-26 17:07:35,336: INFO: cursor: Number of results in first chunk: 1]
[2026-06-26 17:07:35,340: INFO: cursor: query: [use schema CONSUMER_INT_MODEL]]
[2026-06-26 17:07:35,622: INFO: cursor: q

In [4]:
from dataclasses import dataclass

@dataclass(frozen=True)
class CandidateIngestionConfig:
    input_table: str
    input_table_pb: str
    output_table: str
    output_table_cleaned: str
    columns: list[str]
    days_prior: int
    days_nomission: int

@dataclass(frozen=True)
class VacancyIngestionConfig:
    input_table: str
    input_table_pb: str
    output_table: str
    output_table_cleaned: str
    columns: list[str]
    days_prior: int

@dataclass(frozen=True)
class ItalianCitiesIngestionConfig:
    input_file: str
    output_table: str
    string_columns: dict
    numeric_columns: dict

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: str
    database: str
    schema: str
    candidate: CandidateIngestionConfig
    vacancy: VacancyIngestionConfig
    italian_cities: ItalianCitiesIngestionConfig
    desired_locations: dict
    parttime_preferenza_perc: list[int]
    max_tokens: int


In [5]:
from src.onBench.constants import *
from src.onBench.utils.common import read_yaml, read_yaml_sp
from src.onBench import logger
class ConfigurationManager:
    def __init__(
        self,
        session = None,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH,
        stage = STAGE_NAME):

        if(not session):
            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)
            self.schema = read_yaml(schema_filepath)
        else:
            self.config = read_yaml_sp(session, stage, CONFIG_FILE_PATH) #"config_test.yaml")
            self.params = read_yaml_sp(session, stage, PARAMS_FILE_PATH) 
            self.schema = read_yaml_sp(session, stage, SCHEMA_FILE_PATH) 
            

        #create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        schema = self.schema.data_ingestion

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            database=config.database,
            schema=config.schema,

            candidate=CandidateIngestionConfig(
                input_table=config.candidate_tables.input_table,
                input_table_pb=config.candidate_tables.input_table_pb,
                output_table=config.candidate_tables.output_table,
                output_table_cleaned=config.candidate_tables.output_table_cleaned,
                columns = schema.candidate.columns,
                days_prior = schema.candidate.days_prior,
                days_nomission = schema.candidate.days_nomission
            ),

            vacancy=VacancyIngestionConfig(
                input_table=config.vacancy_tables.input_table,
                input_table_pb=config.vacancy_tables.input_table_pb,
                output_table=config.vacancy_tables.output_table,
                output_table_cleaned=config.vacancy_tables.output_table_cleaned,
                columns = schema.vacancy.columns,
                days_prior = schema.vacancy.days_prior

            ),
            italian_cities=ItalianCitiesIngestionConfig(
                input_file=config.italian_cities.input_file,
                output_table=config.italian_cities.output_table,
                string_columns = schema.italian_cities.string_columns,
                numeric_columns = schema.italian_cities.numeric_columns,
            ),

            desired_locations = schema.desired_locations,
            parttime_preferenza_perc = schema.parttime_preferenza_perc,
            max_tokens=schema.max_tokens
        )

        return data_ingestion_config

In [6]:
from snowflake.snowpark.functions import col, coalesce, lit, regexp_replace, when, greatest, length, trim, lower, udf, to_varchar
from snowflake.snowpark.functions import expr, split, iff, to_timestamp, concat
from snowflake.snowpark.types import StructType, StructField, StringType, FloatType
from snowflake.snowpark import Row

class DataIngestion:
    """
    This class is used to create the candidate and vacancie tables.
    It provides methods for:
        - reading the input tables
        - filtering the tables in order to select only the candidates/vacancies that have been created/modified recently
        - cleanings candidates' CVs and descriptions

    """

    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def clean_description(self, df, column = "description_cleaned"):
        """Cleans string columns such as parsed CVs and job descriptions:
            - replaces multiple consecutive whitespaces with a single whitespace (preserves newlines)
            - removes all html tags
            - lowercases all text

        Args:
            df (DataFrame): dataframe
            column (string): column to clean

        Returns:
            df (Snowpark Dataframe): dataframe
        """

        #df = df.filter((col(column).is_not_null()) & (trim(col(column)) != ""))
        
        def build_normalize_whitespace_udf():
            def normalize(text: str) -> str:
                import re
                if text is None:
                    return ''
                return re.sub(r'[ \t]+', ' ', text).strip()

            return udf(normalize, return_type=StringType(), input_types=[StringType()])
        normalize_udf = build_normalize_whitespace_udf()
        df = df.with_column(column, normalize_udf(df[column]))
        

        def build_clean_html_udf():
            from bs4 import BeautifulSoup
            def clean_html(text: str) -> str:
                if not text:
                    return ""
                return BeautifulSoup(text, "html.parser").get_text()

            return udf(
                clean_html,
                return_type=StringType(),
                input_types=[StringType()],
                packages=["beautifulsoup4"]
            )

        clean_html_udf = build_clean_html_udf()
        df = df.with_column(column, clean_html_udf(df[column]))

        df = df.with_column(column, lower(df[column]))

        """
        df = df.filter((col(column).is_not_null()) & 
                       (trim(col(column)) != "") &
                       (length(trim(col(column))) > 5) &
                       (~col(column).like("%None%")) &
                       (~col(column).like("%null%"))
                       )
        """
        df = df.with_column(column, col(column).cast("STRING"))
        
        logger.info(f"Table successfully cleaned")

        return df

    def read_table_candidate(self, session):
        """Creates candidate table with all the columns needed

        Args:
            session (Session): Snowflake session

        Returns:
            df (Snowpark Dataframe): dataframe
        """
        database = self.config.database
        schema = self.config.schema
        input_table = self.config.candidate.input_table
        input_table_pb = self.config.candidate.input_table_pb
        columns_candidate = self.config.candidate.columns.candidate
        columns_joborder = self.config.candidate.columns.joborder
        days_nomission = self.config.candidate.days_nomission

        # Create a temporary table with only 100 rows from the view 
        if(False):
            input_table = f"candidate_materialized_100"
            """
            CREATE OR REPLACE TABLE IT_DISCOVERY.CONSUMER_INT_MODEL.candidate_materialized AS 
            SELECT * FROM IT_DISCOVERY.CONSUMER_INT_MODEL.CANDIDATE_CLEANED -- this is your VIEW 
            WHERE date_added >= DATEADD(day, -60, CURRENT_TIMESTAMP())
            LIMIT 1000
            """

        # Reads only "no mission" candidates
        df = session.sql(f"""
            WITH dati_sospensione AS (
                SELECT *
                FROM (
                    SELECT *,
                        ROW_NUMBER() OVER (
                            PARTITION BY dbsospcandno
                            ORDER BY load_datetimestamp DESC
                        ) AS rn2
                    FROM {database}.{schema}.PB_FTCANDTISOSPENSIONEDISP_CLEANED
                )
                WHERE rn2 = 1
            ),

            cte_candidato AS (
                SELECT
                    ca.dbcandno AS Candidate_Id,
                    ca.dbAssTINoMission,
                    ca.dbAssTIDataCessaz,
                    d.dbSospDescGiust AS desc_giustificativo,
                    j.dbjobordno AS jofilled
                FROM {database}.{schema}.{input_table_pb} ca

                LEFT JOIN dati_sospensione d
                    ON ca.dbcandno = d.dbSospCandNo
                AND (d.dbSospDataFine >= CURRENT_DATE OR d.dbSospDataFine IS NULL)
                AND (d.dbSospDataInizio <= CURRENT_DATE OR d.dbSospDataInizio IS NULL)
                AND d.dbSospDescGiust NOT ILIKE 'mal.no%'

                LEFT JOIN {database}.{schema}.PB_FTJOBORDER_CLEANED j
                    ON j.dbjobcandno = ca.dbcandno
                AND j.dbjobstatus IN (2,4)
                AND j.dbjobenddate >= CURRENT_DATE
                AND j.dbjobdatecommence <= CURRENT_DATE

                WHERE ca.dbAssTINoMission IS NOT NULL
                AND ca.dbAssTINoMission <= DATEADD(DAY, {days_nomission}, CURRENT_DATE)
            ),

            candidate_onbench AS (
                SELECT DISTINCT
                    c.Candidate_Id AS candidate_id_pb,
                    c2.bhid AS candidate_id_bh,

                    CASE
                        WHEN c.dbAssTIDataCessaz <= CURRENT_DATE THEN 'Cessato'
                        WHEN c.dbAssTINoMission > CURRENT_DATE THEN 'Futuro no mission'
                        WHEN c.dbAssTINoMission <= CURRENT_DATE
                            AND c.jofilled IS NULL
                            AND c.desc_giustificativo IS NULL
                            THEN 'No mission'
                        WHEN c.dbAssTINoMission <= CURRENT_DATE
                            AND c.jofilled > 0
                            THEN 'Ricollocato'
                        WHEN c.dbAssTINoMission < CURRENT_DATE
                            AND c.desc_giustificativo IS NOT NULL
                            THEN 'Indisponibile'
                        ELSE 'da controllare'
                    END AS status_candidato

                FROM cte_candidato c

                LEFT JOIN {database}.{schema}.PB_MATRIX_CANDIDATE_CLEANED c2
                    ON c.Candidate_Id = c2.MOID

            ),

            candidates AS (
                SELECT
                    candidateid,
                    pb_id,
                    first_name, 
                    last_name, 
                    skill_set,
                    education_degree,
                    degree_list, 
                    salary_low, 
                    date_available, 
                    will_relocate, 
                    desired_location, 
                    parttime_preferenza_perc, 
                    turno_preferenza,
                    description
                FROM {database}.{schema}.{input_table}
                WHERE isdeleted = 0
                AND is_anonymized = 0
            ),

            ranked AS (
                SELECT
                    dbjobordno,
                    dbjobcandno,
                    dbjobsite,
                    dbjobposition,
                    dbjobdatecommence,
                    dbjobenddate,
                    dbiparttimepcnt,
                    ROW_NUMBER() OVER (
                        PARTITION BY dbjobcandno
                        ORDER BY dbjobdatecommence
                    ) AS rn_first,
                    ROW_NUMBER() OVER (
                        PARTITION BY dbjobcandno
                        ORDER BY dbjobdatecommence DESC
                    ) AS rn_last
                FROM {database}.{schema}.pb_ftjoborder_cleaned
            ),

            jobs_ranked AS (
                SELECT
                    dbjobordno,
                    dbjobcandno,
                    dbjobsite,
                    dbjobposition,
                    dbjobdatecommence,
                    dbjobenddate,
                    dbiparttimepcnt,
                    CASE
                        WHEN rn_first = 1 THEN 'first'
                        WHEN rn_last = 1 THEN 'last'
                        ELSE 'intermediate'
                    END AS job_rank
                FROM ranked
            ),

            jobs_ranked_info AS (
                SELECT
                    jo.dbjobordno,
                    jo.dbjobcandno,
                    jo.job_rank,
                    jo.dbjobsite,
                    jo.dbjobposition,
                    jo.dbjobdatecommence,
                    jo.dbjobenddate,
                    jo.dbiparttimepcnt,
                    je.dbilevel,
                    je.dbiqualification,
                    ccnl_codice,
                    ccnl_descrizione
                FROM jobs_ranked jo
                INNER JOIN {database}.{schema}.pb_ftjobext_cleaned je
                    ON jo.dbjobordno = je.dbjobordno
                LEFT JOIN {database}.{schema}.v_ppccnlsalias cc
                    ON cc.chiave = je.ccnlid
            ),

            jobs_ranked_agg AS (
                SELECT
                    dbjobcandno,

                    MAX(CASE WHEN job_rank = 'first' THEN dbjobsite END) AS job_site,
                    MAX(CASE WHEN job_rank = 'first' THEN dbjobposition END) AS job_first,

                    NULLIF(
                        LISTAGG(
                            CASE WHEN job_rank = 'intermediate' THEN dbjobposition END,
                            ', '
                        ) WITHIN GROUP (ORDER BY dbjobdatecommence),
                        ''
                    ) AS job_intermediate,

                    MAX(CASE WHEN job_rank = 'last' THEN dbjobposition END) AS job_last,

                    MIN(dbjobdatecommence) AS start_date,
                    MAX(dbjobenddate) AS end_date,

                    MAX(CASE WHEN job_rank = 'first' THEN dbiparttimepcnt END) AS parttime_perc,
                    MAX(CASE WHEN job_rank = 'first' THEN dbilevel END) AS ccnl_level,
                    MAX(CASE WHEN job_rank = 'first' THEN dbiqualification END) AS legal_category,
                    MAX(CASE WHEN job_rank = 'first' THEN ccnl_codice END) AS ccnl_code,
                    MAX(CASE WHEN job_rank = 'first' THEN ccnl_descrizione END) AS ccnl_description

                FROM jobs_ranked_info
                GROUP BY dbjobcandno
            )

            SELECT
                {", ".join(f"cc.{source} AS {alias}" for source, alias in columns_candidate.items())},
                IFF(
                    COALESCE(
                        NULLIF(TRIM(description), ''),
                        NULLIF(TRIM(skill_set), ''),
                        NULLIF(TRIM(education_degree), ''),
                        NULLIF(TRIM(degree_list), '')
                    ) IS NULL,
                    NULL,
                    CONCAT(
                        'Curriculum: ', COALESCE(description, ''), ' | ',
                        'Skills: ', COALESCE(skill_set, ''), ' | ',
                        'Titoli di studio richiesti: ', COALESCE(education_degree, ''), ' | ',
                        'Certificazioni: ', COALESCE(degree_list, ''), ' | '
                    )
                ) AS description,
                {", ".join(f"jri.{source} AS {alias}" for source, alias in columns_joborder.items())},
                --cob.status_candidato,
            FROM candidates cc
            INNER JOIN candidate_onbench cob
                ON cc.pb_id = cob.candidate_id_pb
            LEFT JOIN jobs_ranked_agg jri
                ON cc.pb_id = jri.dbjobcandno
            where cob.status_candidato = 'No mission'
            """)

        #candidate_df = candidate_df.with_column("candidateid", col("candidateid").cast("string"))

        logger.info(f"Table {input_table} successfully read")
        
        return df

    def read_table_vacancy(self, session):
        """Creates vacancy table with all the columns needed

        Args:
            session (Session): Snowflake session

        Returns:
            df (Snowpark Dataframe): dataframe
        """
        database = self.config.database
        schema = self.config.schema
        input_table_vacancy = self.config.vacancy.input_table
        columns = self.config.vacancy.columns
        days_prior = self.config.vacancy.days_prior

        # Create a temporary table with only 100 rows from the view 
        if(False):
            input_table_vacancy = f"joborder_materialized_100"
            """
            CREATE OR REPLACE TABLE IT_DISCOVERY.CONSUMER_INT_MODEL.joborder_materialized AS 
            SELECT * FROM IT_DISCOVERY.CONSUMER_INT_MODEL.JOBORDER_CLEANED -- this is your VIEW 
            WHERE isopen = 1 AND status = 'Accepting Candidates' 
            AND datelastmodified >= DATEADD(day, -60, CURRENT_TIMESTAMP())
            LIMIT 1000

            """

        # Reads only "fresh" vacancies, i.e. vacancies that have been added or have been modified  in the last N days
        df = session.sql(f"""
            WITH filtered_jc AS (
                SELECT 
                joborderid,
                jobtitle,
                citta,
                regione,
                salary,
                data_inizio_validita,
                skill_list,
                part_time_percent,
                titoli_richiesti,
                descrizione_uso_interno,
                testo_pubblicazione,
                richieste_aggiuntive,
                requisiti,
                isopen, 
                status, 
                datelastmodified,
                FROM {database}.{schema}.{input_table_vacancy}
                WHERE isopen = 1 AND status = 'Accepting Candidates'
            ),

            jsc_agg AS (
                SELECT
                    joborderid,
                    MAX(data_ultimo_cambio_status) AS last_status_change
                FROM IT_DISCOVERY.CONSUMER_INT_MODEL.JOBSUBMISSION_CLEANED
                WHERE data_ultimo_cambio_status >= DATEADD(day, -{days_prior}, CURRENT_TIMESTAMP())
                GROUP BY joborderid
            )

            SELECT
            {", ".join(f"jc.{source} AS {alias}" for source, alias in columns.items())},
            CASE 
                WHEN jc.datelastmodified IS NULL
                AND jsc_agg.last_status_change IS NULL
                    THEN NULL
                ELSE GREATEST(
                        COALESCE(jc.datelastmodified, TO_TIMESTAMP('0001-01-01 00:00:00')),
                        COALESCE(jsc_agg.last_status_change, TO_TIMESTAMP('0001-01-01 00:00:00'))
                    )
            END AS date_last_active,
            IFF(
                COALESCE(
                    NULLIF(TRIM(skill_list), ''),
                    NULLIF(TRIM(part_time_percent), ''),
                    NULLIF(TRIM(titoli_richiesti), ''),
                    NULLIF(TRIM(DESCRIZIONE_USO_INTERNO), ''),
                    NULLIF(TRIM(TESTO_PUBBLICAZIONE), ''),
                    NULLIF(TRIM(RICHIESTE_AGGIUNTIVE), ''),
                    NULLIF(TRIM(REQUISITI), '')
                ) IS NULL,
                
                NULL,
                
                CONCAT(
                    'Skills richieste: ', COALESCE(skill_list, ''), ' | ',
                    'Percentuale part time: ', COALESCE(part_time_percent, ''), ' | ',
                    'Titoli di studio richiesti: ', COALESCE(titoli_richiesti, ''), ' | ',
                    'DESCRIZIONE USO INTERNO: ', COALESCE(DESCRIZIONE_USO_INTERNO, ''), ' | ',
                    'TESTO PUBBLICAZIONE: ', COALESCE(TESTO_PUBBLICAZIONE, ''), ' | ',
                    'RICHIESTE AGGIUNTIVE: ', COALESCE(RICHIESTE_AGGIUNTIVE, ''), ' | ',
                    'REQUISITI: ', COALESCE(REQUISITI, '')
                )
            ) AS description
            FROM filtered_jc jc
            LEFT JOIN jsc_agg
                ON jc.joborderid = jsc_agg.joborderid
            WHERE
                jc.datelastmodified >= DATEADD(day, -{days_prior}, CURRENT_TIMESTAMP())
                OR jsc_agg.last_status_change >= DATEADD(day, -{days_prior}, CURRENT_TIMESTAMP())

        """)

        logger.info(f"Table {input_table_vacancy} successfully read")

        return df


    
    def clean_table_candidate(self, session):
        """Feature engineering and cleaning of the candidate table:
            - PARTTIME_PERC: if missing, default value is 100; if it's 0, it's set to None
            - Removes leading and trailing whitespaces from all string columns
            - Removes very long meaningless sequences of characters
            - General cleaning of the description column:
                - replaces multiple consecutive whitespaces with a single whitespace (preserves newlines)
                - removes all html tags
                - lowercases all text
            - Removes descriptions that are longer than max_tokens characters

        Args:
            session (Session): Snowflake session

        Returns:
            df (Snowpark Dataframe): dataframe
        """
        database = self.config.database
        schema = self.config.schema
        input_table = self.config.candidate.output_table
        max_tokens = self.config.max_tokens
        desired_locations = self.config.desired_locations
        parttime_preferenza_perc = self.config.parttime_preferenza_perc

        df = session.table(f"{database}.{schema}.{input_table}")  

        df = df.with_column("description_raw", col("description"))
        
        #Desired location is set to its default values: "Locale", "Regionale", "Nazionale", "Internazionale". Only one value allowed
        case_parts = []
        for level in reversed(desired_locations):   # highest first
            case_parts.append(f"WHEN DESIRED_LOCATION ILIKE '%{level}%' THEN '{level}'")

        case_expr = "CASE " + " ".join(case_parts) + " ELSE NULL END"

        #Partime percentage as of bullhorn source. Recruiter is supposed to insert the value with a drow down menu with set values.
        #In reality it's more similar to a free text input where you'll find values such ash "50", "50%", "50, 60" etc
        #Parttime preferenza perc is set to its default values representing percentages: [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]. If missing, default value is 100
        df = df.with_column(
            "PARTTIME_PREFERENZA_PERC",
            coalesce(col("PARTTIME_PREFERENZA_PERC"), lit("100%"))
        )

        #it can contain multiple values such as "80, 100". If so, it will take the maximum value.
        exprs = [
            when(col("PARTTIME_PREFERENZA_PERC").contains(lit(v)), lit(int(v))).otherwise(lit(0))
            for v in parttime_preferenza_perc
        ]
        # Take the maximum across all expressions
        df = df.with_column(
            "PARTTIME_PREFERENZA_PERC",
            greatest(*exprs)
        )
        #If it contained none of the default values, it's set to None
        df = df.with_column(
            "PARTTIME_PREFERENZA_PERC",
            when(col("PARTTIME_PREFERENZA_PERC") == 0, None).otherwise(col("PARTTIME_PREFERENZA_PERC"))
        )

        df = df.with_column(
            "DESIRED_LOCATION",
            expr(case_expr)
        )
        #Turno preferenza is set to its default values: ["Mattina", "Pomeriggio", "Notte", "Fine Settimana"] Multiple values are allowed as a string of comma separated values such as "Mattina, Pomeriggio"
        df = df.with_column(
            "turno_preferenza",
            split(col("turno_preferenza"), lit(","))
        )
        

        
        #Partime percentage as of powerbase source. Integer value between 0 and 100
        #If missing, default value is 100
        df = df.with_column(
            "PARTTIME_PERC",
            coalesce(col("PARTTIME_PERC"), lit(100))
        )
        #If it's 0, it's set to None
        df = df.with_column(
            "PARTTIME_PERC",
            when(col("PARTTIME_PERC") == 0, None).otherwise(col("PARTTIME_PERC"))
        )

        df = df.with_column("CCNL_CODE", to_varchar(col("CCNL_CODE").cast("int")))


        # Trim all leading and trailing whitespaces from all string columns
        string_cols = [
            f.name
            for f in df.schema.fields
            if f.datatype.__class__.__name__ == "StringType"
        ]
        for c in string_cols:
            df = df.with_column(c, trim(col(c)))

        

        #Removes very long meaningless sequences of characters, including character codes for attachments (pdf, jpg etc)
        noise_pattern = r"(/9j/[A-Za-z0-9+/=]{30,}|iVBORw0KGgo[A-Za-z0-9+/=]{30,}|JVBERi0x[A-Za-z0-9+/=]{30,}|[A-Za-z0-9+/=]{80,}|[^A-Za-z0-9 ]{30,}|([A-Za-z]{3,} ){15,})"
        df = df.with_column("description", regexp_replace(col("description"), noise_pattern, "") )

        df = self.clean_description(df, "description")

        #If description is longer than max_tokens, replace it with an empty string
        df = (
            df
            .with_column(
                "description",
                when(
                    length(col("description")) > max_tokens,
                    lit("")                     
                ).otherwise(col("description"))
            )
        )

        logger.info(f"Table {input_table} successfully cleaned")

        return df

    def clean_table_vacancy(self, session):
        """Feature engineering and cleaning of the vacancy table:
            - Removes leading and trailing whitespaces from all string columns
            - Removes very long meaningless sequences of characters
            - General cleaning of the description column:
                - replaces multiple consecutive whitespaces with a single whitespace (preserves newlines)
                - removes all html tags
                - lowercases all text
            - Removes descriptions that are longer than max_tokens characters

        Args:
            session (Session): Snowflake session

        Returns:
            df (Snowpark Dataframe): dataframe
        """
        database = self.config.database
        schema = self.config.schema
        input_table = self.config.vacancy.output_table
        max_tokens = self.config.max_tokens


        df = session.table(f"{database}.{schema}.{input_table}")

        string_cols = [
            f.name
            for f in df.schema.fields
            if f.datatype.__class__.__name__ == "StringType"
        ]
        for c in string_cols:
            df = df.with_column(c, trim(col(c)))


        df = df.with_column("description_raw", col("description"))

        #Removes very long meaningless sequences of characters
        noise_pattern = r"(/9j/[A-Za-z0-9+/=]{30,}|iVBORw0KGgo[A-Za-z0-9+/=]{30,}|JVBERi0x[A-Za-z0-9+/=]{30,}|[A-Za-z0-9+/=]{80,}|[^A-Za-z0-9 ]{30,}|([A-Za-z]{3,} ){15,})"
        df = df.with_column("description", regexp_replace(col("description"), noise_pattern, "") )
        df = self.clean_description(df, "description")

        df = (
            df
            .with_column(
                "description",
                when(
                    length(col("description")) > max_tokens,
                    lit("")                      # replace with empty string
                ).otherwise(col("description"))   # keep original
            )
        )

        logger.info(f"Table {input_table} successfully cleaned")

        return df

    def read_cities_file(self, session):
        """Reads XLSX file containing italian cities and writes table in Snowflake

        Args:
            session (Session): Snowflake session

        Returns:
            df (Snowpark Dataframe): dataframe
        """
        italian_cities_file = self.config.italian_cities.input_file
        string_columns = self.config.italian_cities.string_columns
        numeric_columns = self.config.italian_cities.numeric_columns

        df = pd.read_excel(italian_cities_file, header=0)

        #Renames columns for consistency
        df.columns = (
            df.columns
            .str.strip()
            .str.replace(" ", "_")
            .str.replace('"', '')
            .str.replace("'", '')
            .str.lower()
        )
        df = df[string_columns + numeric_columns]
        
        # Converts ZIP to string (preserve leading zeros)
        df["zip"] = df["zip"].apply(lambda x: str(int(x)).zfill(5) if pd.notnull(x) else None)
        
        # Converts string columns
        for col in string_columns:
            df[col] = df[col].astype(str).str.strip()

        # Converts latitude and longitude to float, handle NaNs
        for col in numeric_columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        # Remove rows with missing city_name
        df = df[
            df["city_name"].notna() &  # Remove NaN and None
            (df["city_name"].str.strip() != "") &  # Remove empty and whitespace-only strings
            (df["city_name"].str.lower().str.strip() != "null") &  # Remove "NULL" string
            (df["city_name"].str.lower().str.strip() != "nan")  # Remove "nan" string
        ]

        #These are necessary in order to avoid columns names with quotes (e.g. "city" instead of city)
        rows = [Row(**row) for row in df.to_dict(orient="records")]
        schema = StructType([
            StructField("unique_identifier", StringType()),
            StructField("city_name", StringType()),
            StructField("province", StringType()),
            StructField("province_ext", StringType()), 
            StructField("region", StringType()), 
            StructField("zip", StringType()),
            StructField("latitude", FloatType()),
            StructField("longitude", FloatType())
        ])

        logger.info(f"XLSX file containing italian cities successfully read")

        df = session.create_dataframe(rows, schema=schema)

        return df
    
    def complete_cities_table(self, session):
        """Uses available information to fill latitude and longitude by prompting a LLM 

        Args:
            session (Session): Snowflake session

        Returns:
            df (Snowpark Dataframe): dataframe
        """

        output_table_italian_cities = self.config.italian_cities.output_table

        df=session.table(output_table_italian_cities)

        missing_df = df.filter(
            col("latitude").is_null() |
            col("longitude").is_null() |
            (lower(col("latitude").cast("string")) == lit("nan")) |
            (lower(col("longitude").cast("string")) == lit("nan")) |
            (col("latitude").cast("string") == lit("NULL")) |
            (col("longitude").cast("string") == lit("NULL")) |
            (~col("latitude").cast("string").rlike(r"^-?\d{1,3}\.\d{1,10}$")) |
            (~col("longitude").cast("string").rlike(r"^-?\d{1,3}\.\d{1,10}$"))
        )


        prompt = concat( 
            lit("Restituisci la latitudine per la seguente città italiana:\n"), 

            lit("Città: "), col("city_name"), lit("\n"), lit("Provincia: "), col("province_ext"), lit("\n"), lit("Regione: "), col("region"), lit("\n"), lit("Nazione: Italia\n\n"), 
            lit("Rispondi SOLO con un numero nel formato seguente: 40.1234567)."),
            lit("Se non conosci la latitudine della città, non restituire nulla") 
            )
        missing_df = missing_df.with_column("latitude",
                            call_function(
                                "SNOWFLAKE.CORTEX.COMPLETE",
                                lit("claude-4-sonnet"),
                                prompt
                                )
                            )

        prompt = concat( 
            lit("Restituisci la longitudine per la seguente città italiana:\n"), 
            lit("Città: "), col("city_name"), lit("\n"), lit("Provincia: "), col("province"), lit("\n"), lit("Regione: "), col("region"), lit("\n"), lit("Nazione: Italia\n\n"), 
            lit("Rispondi SOLO con un numero nel formato seguente: 10.1234567)."),
            lit("Se non conosci la longitudine della città, non restituire nulla")  
            )
        missing_df = missing_df.with_column("longitude",
                            call_function(
                                "SNOWFLAKE.CORTEX.COMPLETE",
                                lit("claude-4-sonnet"),
                                prompt
                                )
                            )
        
        
        #Makes sure latitude and longitude respect the correct format
        missing_df = missing_df.with_column(
            "latitude",
            when(
                col("latitude").cast("string").rlike(r"^-?\d{1,3}\.\d{1,10}$"),
                col("latitude")
            ).otherwise(lit(None))
        ).with_column(
            "longitude",
            when(
                col("longitude").cast("string").rlike(r"^-?\d{1,3}\.\d{1,10}$"),
                col("longitude")
            ).otherwise(lit(None))
        )
        

        complete_df = df.filter(
            ~(
                col("latitude").is_null() |
                col("longitude").is_null() |
                (lower(col("latitude").cast("string")) == lit("nan")) |
                (lower(col("longitude").cast("string")) == lit("nan")) |
                (col("latitude").cast("string") == lit("NULL")) |
                (col("longitude").cast("string") == lit("NULL")) |
                (~col("latitude").cast("string").rlike(r"^-?\d{1,3}\.\d{1,10}$")) |
                (~col("longitude").cast("string").rlike(r"^-?\d{1,3}\.\d{1,10}$"))
            )
        )

        final_df = complete_df.union_by_name(missing_df)                   

        return final_df 
    
    def write_table(self, df, table_name = 'output_table'):
        """Writes dataframe in Snowflake

        Args:
            df (Snowpark Dataframe): dataframe
            table_name (string): name of the Snowflake table
        """

        df.write.save_as_table(table_name, mode="overwrite")
        logger.info(f"Table {table_name} successfully written")




In [7]:
try:
    """
    if self.deployment:
        config = ConfigurationManager(session)
    else:
        config = ConfigurationManager()
    """    
    config = ConfigurationManager() 
    
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)

    
    #df = data_ingestion.read_table_vacancy(session)
    #data_ingestion.write_table(df, data_ingestion.config.vacancy.output_table)
    #df = data_ingestion.clean_table_vacancy(session)
    #data_ingestion.write_table(df, data_ingestion.config.vacancy.output_table_cleaned)

    #df = data_ingestion.read_table_candidate(session)
    #data_ingestion.write_table(df, data_ingestion.config.candidate.output_table)
    df = data_ingestion.clean_table_candidate(session)
    data_ingestion.write_table(df, data_ingestion.config.candidate.output_table_cleaned)
    
    
except Exception as e:
    raise e

[2026-06-26 17:07:35,818: INFO: common: yaml file: C:\Users\fiscarelli\Desktop\Progetti\Manpower IT\OnBench\On-Bench-Candidates-to-Jobs-Auto-Match\config\config_test.yaml loaded successfully]
[2026-06-26 17:07:35,822: INFO: common: yaml file: C:\Users\fiscarelli\Desktop\Progetti\Manpower IT\OnBench\On-Bench-Candidates-to-Jobs-Auto-Match\config\params.yaml loaded successfully]
[2026-06-26 17:07:35,842: INFO: common: yaml file: C:\Users\fiscarelli\Desktop\Progetti\Manpower IT\OnBench\On-Bench-Candidates-to-Jobs-Auto-Match\config\schema.yaml loaded successfully]
[2026-06-26 17:07:35,845: INFO: cursor: query: [SELECT  *  FROM IT_DISCOVERY.CONSUMER_INT_MODEL.MPG_IT_ONBENCH_TEST_CANDIDATE]]
[2026-06-26 17:07:36,257: INFO: cursor: query execution done]
[2026-06-26 17:07:36,257: INFO: cursor: Number of results in first chunk: 0]
[2026-06-26 17:07:36,286: INFO: cursor: query: [SELECT "CANDIDATEID", "PB_ID", "FIRST_NAME", "LAST_NAME", "SKILLS", "EDUCATION",...]]
[2026-06-26 17:07:36,791: INFO: c